# 03 Anatomy setup

This notebook prepares MNE anatomy assets after FreeSurfer recon-all:

- watershed BEM surfaces
- dense scalp surfaces for coregistration/QC
- BEM model and BEM solution
- surface source space
- optional source-space distances
- optional volume source space
- optional fsaverage label morphing

This is still separate from the MEG preprocessing notebooks. The forward/inverse/source pipeline comes later.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.workflow import existing_output_policy_for_step, should_overwrite
from meeg_pipeline.anatomy import (
    anatomy_status_to_dataframe,
    apply_watershed_bem,
    bem_solution_path,
    compute_source_space_distances,
    fetch_fsaverage_parcellations,
    make_bem_model_and_solution,
    make_dense_scalp_surfaces,
    morph_labels_from_fsaverage,
    resolve_subjects,
    results_to_dataframe,
    setup_surface_source_space,
    setup_volume_source_space,
)

def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)


In [ ]:
SUBJECTS = "all"
OVERWRITE_STEPS = []

RUN_WATERSHED = True
RUN_DENSE_SCALP = True
RUN_BEM = True
RUN_SOURCE_SPACE = True
RUN_SOURCE_DISTANCES = config.anatomy.source_space.add_dist is True
RUN_VOLUME_SOURCE_SPACE = config.anatomy.volume_source_space.enabled
RUN_LABEL_MORPH = True
FETCH_FSAVERAGE_PARCELLATIONS = True

policies = pd.DataFrame([
    {"step": step, "overwrite": should_overwrite(step, OVERWRITE_STEPS), "policy": existing_output_policy_for_step(step, OVERWRITE_STEPS)}
    for step in [
        "watershed",
        "dense_scalp",
        "bem",
        "source_space",
        "source_distances",
        "volume_source_space",
        "morph_labels",
    ]
])
policies

In [ ]:
selected_subjects = resolve_subjects(
    SUBJECTS,
    subjects_dir=config.freesurfer.subjects_dir,
)

selected_subjects

In [ ]:
anatomy_status_to_dataframe(
    selected_subjects,
    subjects_dir=config.freesurfer.subjects_dir,
    mri_root=config.paths.mri_root,
    t1_patterns=config.anatomy.t1_patterns,
    t2_patterns=config.anatomy.t2_patterns,
    spacing=config.anatomy.source_space.spacing,
    bem_ico=config.anatomy.bem.ico,
    bem_conductivity=config.anatomy.bem.conductivity,
)

In [ ]:
setup_results = []

if RUN_WATERSHED:
    for subject in selected_subjects:
        setup_results.append(
            apply_watershed_bem(
                subject,
                subjects_dir=config.freesurfer.subjects_dir,
                freesurfer_home=config.freesurfer.home,
                volume=config.anatomy.watershed.volume,
                overwrite=should_overwrite("watershed", OVERWRITE_STEPS),
            )
        )

if RUN_DENSE_SCALP:
    for subject in selected_subjects:
        setup_results.append(
            make_dense_scalp_surfaces(
                subject,
                subjects_dir=config.freesurfer.subjects_dir,
                freesurfer_home=config.freesurfer.home,
                overwrite=should_overwrite("dense_scalp", OVERWRITE_STEPS),
            )
        )

if RUN_BEM:
    for subject in selected_subjects:
        setup_results.append(
            make_bem_model_and_solution(
                subject,
                subjects_dir=config.freesurfer.subjects_dir,
                ico=config.anatomy.bem.ico,
                conductivity=config.anatomy.bem.conductivity,
                on_existing=existing_output_policy_for_step("bem", OVERWRITE_STEPS),
            )
        )

if RUN_SOURCE_SPACE:
    for subject in selected_subjects:
        setup_results.append(
            setup_surface_source_space(
                subject,
                subjects_dir=config.freesurfer.subjects_dir,
                spacing=config.anatomy.source_space.spacing,
                surface=config.anatomy.source_space.surface,
                add_dist=config.anatomy.source_space.add_dist,
                n_jobs=config.runtime.n_jobs,
                on_existing=existing_output_policy_for_step("source_space", OVERWRITE_STEPS),
            )
        )

if RUN_SOURCE_DISTANCES:
    for subject in selected_subjects:
        setup_results.append(
            compute_source_space_distances(
                subject,
                subjects_dir=config.freesurfer.subjects_dir,
                spacing=config.anatomy.source_space.spacing,
                n_jobs=config.runtime.n_jobs,
                on_existing=existing_output_policy_for_step("source_distances", OVERWRITE_STEPS),
            )
        )

if RUN_VOLUME_SOURCE_SPACE:
    for subject in selected_subjects:
        setup_results.append(
            setup_volume_source_space(
                subject,
                subjects_dir=config.freesurfer.subjects_dir,
                bem_path=bem_solution_path(
                    config.freesurfer.subjects_dir,
                    subject,
                    ico=config.anatomy.bem.ico,
                    conductivity=config.anatomy.bem.conductivity,
                ),
                pos=config.anatomy.volume_source_space.spacing,
                on_existing=existing_output_policy_for_step("volume_source_space", OVERWRITE_STEPS),
            )
        )

if RUN_LABEL_MORPH:
    if FETCH_FSAVERAGE_PARCELLATIONS:
        fetch_fsaverage_parcellations(
            subjects_dir=config.freesurfer.subjects_dir,
            parcellations=config.anatomy.labels.parcellations,
            freesurfer_home=config.freesurfer.home,
        )
    for subject in selected_subjects:
        setup_results.extend(
            morph_labels_from_fsaverage(
                subject,
                subjects_dir=config.freesurfer.subjects_dir,
                parcellations=config.anatomy.labels.parcellations,
                overwrite=should_overwrite("morph_labels", OVERWRITE_STEPS),
            )
        )

results_to_dataframe(setup_results)

In [ ]:
anatomy_status_to_dataframe(
    selected_subjects,
    subjects_dir=config.freesurfer.subjects_dir,
    mri_root=config.paths.mri_root,
    t1_patterns=config.anatomy.t1_patterns,
    t2_patterns=config.anatomy.t2_patterns,
    spacing=config.anatomy.source_space.spacing,
    bem_ico=config.anatomy.bem.ico,
    bem_conductivity=config.anatomy.bem.conductivity,
)